# 01 · Joint training: the non-continual reference

**Main question.** How well can polynomial models learn when all training data are available together?

This notebook is the *reference* for `02_continual_learning.ipynb`. Joint training sees every experience (binary part) or every class (Split MNIST part) at once, so it is **not** a continual-learning experiment: there is no sequence of experiences, nothing can be forgotten, and **no forgetting is computed here**.

| Section | Question | Research question |
|---|---|---|
| 1 | What do the synthetic targets look like? | RQ1 |
| 2 | How does polynomial degree affect accuracy? | RQ1 |
| 3 | Does more training help, and does it depend on degree? | RQ1 |
| 4 | What does joint training reach on Split MNIST? | RQ6 reference |
| 5 | Summary of what was (and was not) established | |

Everything below calls package code (`polynomial_cl`); nothing is read from saved result files. Section 4 needs the MNIST files in `data/mnist/` (downloaded on first use if the network is available) and is skipped with a message otherwise.

## Configuration

In [ ]:
SEED = 0
N_TRAIN, N_TEST = (
    200,
    500,
)  # per experience; the 10 experiences are pooled for joint training
LEARNING_RATE = 0.5  # binary models (full-batch gradient descent)
DEGREES = (1, 2, 3, 4, 6, 8, 10)
EPOCHS = 5000  # Section 2
SHOWN_DEGREES = (1, 2, 4, 10)  # decision regions drawn in Section 2
BUDGET_DEGREES = (2, 4, 10)  # Section 3
BUDGETS = (100, 500, 2000, 5000)
MNIST_DEGREES, MNIST_EPOCHS, MNIST_SEEDS, MNIST_LR = (
    (1, 2, 3),
    (3, 10, 30),
    (0, 1, 2),
    0.1,
)

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Works from the repository root or from notebooks/ (locates the repository to find data/mnist).
ROOT = next(
    r for r in (Path.cwd(), Path.cwd().parent) if (r / "src" / "polynomial_cl").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

from polynomial_cl import metrics
from polynomial_cl.binary import combine_splits, make_experiences, run_joint_study
from polynomial_cl.continual import make_feature_data, run_joint_comparison
from polynomial_cl.data import (
    circular_target,
    load_projected_mnist,
    make_synthetic_dataset,
    sinusoidal_target,
)
from polynomial_cl.plotting import synthetic as P

TARGETS = {"sinusoidal": sinusoidal_target, "circular": circular_target}

## 1 · Synthetic target functions

**Question.** What decision geometries do the binary experiments use, and is the data interface the same for both?

**Why it matters.** RQ1 asks whether polynomial models can learn different functions. The generator is shared; only the *target function* changes (it defines how labels are produced, not how experiences are cut out).

**Experiment.** Draw reproducible samples from two targets on $[-1,1]^2$ with `make_synthetic_dataset`:
the existing **sinusoid** ($y=1$ when $x_2 > 0.5\sin(3x_1)$) and a **circle** ($y=1$ inside radius 0.7 around the origin). The circle is a geometry demonstration, not a new benchmark. The cell also checks that the sinusoidal experiences used below are exactly those of the unchanged default protocol.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, (name, target) in zip(axs, TARGETS.items()):
    x, y = make_synthetic_dataset(target, n_samples=400, seed=SEED)
    P.plot_ground_truth(
        ax, samples=(x, y), target=target, title=f"{name} target (400 samples)"
    )
    print(f"{name:10s} class-1 share = {y.mean():.2f}")
plt.tight_layout()
plt.show()

experiences = {
    "sinusoidal": make_experiences(n_train=N_TRAIN, n_test=N_TEST, seed=SEED),
    "circular": make_experiences(
        n_train=N_TRAIN, n_test=N_TEST, seed=SEED, target_function=circular_target
    ),
}
default = make_experiences(n_train=N_TRAIN, n_test=N_TEST, seed=SEED)
same_inputs = all(
    np.array_equal(a.train.x, b.train.x) and np.array_equal(a.train.y, b.train.y)
    for a, b in zip(experiences["sinusoidal"], default)
)
print("sinusoidal experiences identical to the default protocol:", same_inputs)
print(
    "pooled training samples per target:",
    sum(len(e.train.y) for e in experiences["circular"]),
)

**Observed.** Both targets are drawn through the same interface. The sinusoid puts about half of the samples in class 1 (0.48 in this draw); the circle about a third (0.36; the circle covers $\pi\cdot0.7^2/4\approx0.385$ of the domain). The sinusoidal experiences used in the rest of the notebook are identical to the ones the unchanged default protocol produces (`True` above).

**Interpretation.** Changing the target only changes how labels are assigned; inputs, sampling streams and the experience strips are untouched, so the sinusoid reproduces the existing experiment and the circle is a like-for-like second geometry. Nothing about learning is claimed in this section.

## 2 · Polynomial degree

**Question.** How does polynomial degree affect the model's ability to approximate the target function?

**Why it matters.** Degree sets the model's capacity ($\binom{d+2}{2}$ coefficients in two variables). Later, expansion adds exactly these dimensions one degree at a time.

**Experiment.** `run_joint_study` pools the training samples of all 10 experiences and trains a fresh binary model per degree with the *same* data, learning rate, seed and number of epochs. Accuracy is measured on every experience's fixed test split (500 samples each) and averaged. Same protocol for both targets.

**Configuration.** Degrees and epochs as set at the top; full-batch gradient descent, learning rate 0.5, no label noise.

In [ ]:
study = {
    name: run_joint_study(exps, DEGREES, [EPOCHS], LEARNING_RATE, seed=SEED)
    for name, exps in experiences.items()
}
for name, rows in study.items():
    print(f"\n{name} target, {EPOCHS} epochs")
    print(
        metrics.format_table(
            [
                {
                    k: r[k]
                    for k in (
                        "degree",
                        "n_features",
                        "final_loss",
                        "mean_accuracy",
                        "min_accuracy",
                    )
                }
                for r in rows
            ]
        )
    )

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for ax, (name, rows) in zip(axs, study.items()):
    P.plot_accuracy_vs_degree(rows, ax=ax, title=f"{name}: accuracy by degree")
plt.tight_layout()
plt.show()

for name, rows in study.items():
    shown = [r["model"] for r in rows if r["degree"] in SHOWN_DEGREES]
    x, y, _ = combine_splits([e.train for e in experiences[name]])
    fig = P.plot_degree_comparison(shown, samples=(x, y), target=TARGETS[name])
    fig.suptitle(
        f"{name}: predicted regions (solid = learned boundary, dashed grey = true boundary)",
        y=1.02,
    )
    plt.show()

**Observed (one seed, 5000 epochs).**

* *Sinusoid:* mean accuracy is 0.908 at degrees 1 and 2, 0.983 at degrees 3 and 4, 0.990 at degree 6 and 0.987 at degrees 8 and 10. The worst experience improves from 0.856 (degree 1) to 0.980 (degrees 8 and 10). Degrees 1 and 2 reach the *same* loss (0.184), and so do degrees 3 and 4 (0.076).
* *Circle:* degree 1 gives 0.612, with its worst experience at 0.302, because the learned region is a single class everywhere (see the figure). Degree 2 gives 0.997; higher degrees are slightly lower (0.996, 0.989, 0.987, 0.986 for degrees 3, 4, 6, 8; 0.986 for 10).

**Interpretation.** The same polynomial machinery represents both geometries once its degree is high enough: a few degrees for the sinusoid, and degree 2 for the circle. A circle is a degree-2 curve, so that is consistent with it. The identical losses of degrees 1/2 and 3/4 on the sinusoid are consistent with the sinusoid score being odd under point reflection while even-degree monomials are even functions, so they add no useful direction; this explanation was **not** tested here. The small decline of the circle accuracy above degree 2 is of the size of the evaluation noise and should not be read as higher degree being worse.

**Caveats.** One dataset draw and one seed, fixed learning rate, no tuning. The pooled test set has 5000 points, so binomial sampling error alone is roughly ±0.004 at 0.91 accuracy and ±0.0015 at 0.99. At 5000 epochs the higher-degree models had not stopped improving (Section 3), so this comparison mixes capacity with convergence.

## 3 · Training budget

**Question.** Does additional training improve the result, and does the effect depend on polynomial degree?

**Why it matters.** Fixed-budget comparisons can mistake "not trained long enough" for "not expressive enough".

**Experiment.** The same joint-training protocol as Section 2 (same pooled data, learning rate and seed), changing only the number of epochs. Each budget is a separate model trained from scratch.

**Configuration.** Degrees and budgets as set at the top.

In [ ]:
budget = {
    name: run_joint_study(exps, BUDGET_DEGREES, BUDGETS, LEARNING_RATE, seed=SEED)
    for name, exps in experiences.items()
}
fig, axs = plt.subplots(2, 2, figsize=(10, 7))
for j, (name, rows) in enumerate(budget.items()):
    P.plot_accuracy_vs_budget(rows, ax=axs[0, j], title=f"{name}: accuracy by budget")
    longest = [r for r in rows if r["epochs"] == max(BUDGETS)]
    for r in longest:
        P.plot_loss_curve(
            r["loss_history"],
            ax=axs[1, j],
            label=f"degree {r['degree']}",
            title=f"{name}: training loss ({max(BUDGETS)} epochs)",
        )
plt.tight_layout()
plt.show()

for name, rows in budget.items():
    print(f"\n{name} target")
    print(
        metrics.format_table(
            [
                {k: r[k] for k in ("degree", "epochs", "final_loss", "mean_accuracy")}
                for r in rows
            ]
        )
    )

**Observed.**

* *Sinusoid, degree 2:* 0.900, 0.903, 0.908, 0.908 for 100, 500, 2000, 5000 epochs, and the loss is the same (0.184) at 2000 and 5000. It has stopped improving.
* *Sinusoid, degree 4:* 0.911, 0.933, 0.968, 0.983; degree 10: 0.930, 0.961, 0.981, 0.987. For both, the loss is still falling at 5000 epochs (degree 4: 0.106 → 0.076; degree 10: 0.082 → 0.061).
* *Circle:* degree 2 goes 0.926 → 0.993 → 0.996 → 0.997; degree 4: 0.981 → 0.985 → 0.988 → 0.989; degree 10: 0.980 → 0.978 → 0.983 → 0.986. At 100 epochs degrees 4 and 10 are ahead of degree 2; from 500 epochs degree 2 is ahead.

**Interpretation.** More training helps, and how much depends on degree. A model whose capacity is too small for the target (sinusoid, degree 2) plateaus: extra epochs are wasted. Higher-degree models keep improving through 5000 epochs, so their Section 2 accuracies are partly limited by training time, not only by capacity. For the circle, the larger models start faster but are not better once trained. That is a statement about this optimiser and budget; an explanation (for example the differing scales of high-degree monomials) was not tested.

**Caveats.** One seed and one learning rate; each budget is a separate run from the same initial point, not a snapshot of one longer run. Differences of about 0.005 or less are within evaluation noise.

## 4 · Split MNIST joint-training reference

**Question.** What performance is reached when the model can learn from all ten classes together?

**Why it matters.** `02_continual_learning.ipynb` trains on the same features, but with the classes arriving in five experiences. This section gives the *joint* reference for that comparison. Joint training is a reference condition, **not** a continual-learning strategy, and its accuracy is not a proven upper bound for the sequential runs (different optimisation paths and budgets).

**Experiment.** Split MNIST images are normalised and mapped to 16 dimensions by a fixed random orthonormal projection (never fitted to data); degree-$d$ scaled polynomial features feed a ten-class softmax head. `run_joint_comparison` trains on all classes at once (`run_joint_reference`) for each degree and epoch budget over three seeds and reports test accuracy and the per-class accuracy. No forgetting is computed.

**Configuration.** Degrees, epochs, seeds and learning rate as set at the top; projection seed 0.

In [ ]:
try:
    train_x, train_y, test_x, test_y = load_projected_mnist(
        ROOT / "data" / "mnist", download=True
    )
except OSError as error:  # no local data and no network: skip this section only
    train_x = None
    print("Split MNIST unavailable, skipping Section 4:", type(error).__name__, error)

if train_x is not None:
    data = make_feature_data(train_x, train_y, test_x, test_y)
    print(
        f"{len(train_y)} training / {len(test_y)} test images, 16 projected dimensions"
    )
    mnist_rows = run_joint_comparison(
        data, MNIST_DEGREES, MNIST_EPOCHS, MNIST_SEEDS, MNIST_LR
    )
    print(
        metrics.format_table(
            [
                {
                    "degree": r["degree"],
                    "epochs": r["epochs"],
                    "features": r["n_features"],
                    "mean_acc": r["test_accuracy_mean"],
                    "std_acc": r["test_accuracy_std"],
                }
                for r in mnist_rows
            ]
        )
    )
    best = max(mnist_rows, key=lambda r: r["test_accuracy_mean"])
    print(
        f"\nper-class test accuracy of the best configuration "
        f"(degree {best['degree']}, {best['epochs']} epochs, mean over seeds):"
    )
    print(
        metrics.format_table(
            [
                {"class": c, "accuracy": a}
                for c, a in enumerate(best["per_class_accuracy_mean"])
            ]
        )
    )

**Observed.** Test accuracy rises with both degree and training budget in every cell of the grid: degree 1 goes from 0.640 to 0.682, degree 2 from 0.683 to 0.754 and degree 3 from 0.713 to 0.788 as epochs go from 3 to 30. The best configuration is degree 3 (969 features) at 30 epochs with 0.788. In it, class 1 is the easiest (0.968) and classes 5 and 8 the hardest (0.664 and 0.662).

**Interpretation.** On these 16-dimensional projected features, polynomial expansion gives the joint model more useful capacity, and none of the configurations has converged by 30 epochs, so these numbers are reference points at a stated budget, **not** ceilings. They are the joint counterpart of the Split MNIST runs in notebook 02, which see the same features with classes arriving in sequence; they say nothing about forgetting.

**Caveats.** The seed standard deviations (≤ 0.001) only reflect different mini-batch orders; the projection and data are fixed, so they understate the uncertainty a different projection would add. Accuracy of about 0.79 is far below what a full-resolution MNIST classifier reaches because of the 16-dimensional projection.

## 5 · Summary

Only what the experiments above established is summarised here.

**Observed results.** (1) The sinusoid and circle targets share one generator, and the sinusoid reproduces the existing experiment. (2) On the sinusoid, joint accuracy rose from 0.908 (degrees 1–2) to about 0.99 (degree ≥ 6); on the circle degree 2 was enough (0.997) and degree 1 failed (0.612). (3) The training budget mattered: a too-small model plateaued, larger ones were still improving at 5000 epochs. (4) On Split MNIST, joint accuracy grew with degree and epochs up to 0.788 (degree 3, 30 epochs).

**Interpretation.** Polynomial models of sufficient degree can represent both geometries, and extra capacity is only useful if it is trained long enough. This supports RQ1 for these two targets; it does not show that polynomial models learn arbitrary targets.

**Limitations.** One data draw and one seed for the binary studies; one learning rate; the MNIST results use a 16-dimensional projection and a short budget; no hyperparameter search was run.

**Left for the continual-learning notebook.** Whether expansion preserves what was learned, how the strategies trade accuracy against forgetting, and whether the Split MNIST findings hold under sequential class arrival.